In [2]:
# %% [markdown]
# Feature Extraction and Classification of Acoustic Signals

# Adapting the 10-class environmental sound architecture to 2-class mine-impact classification while preserving:
# 1. **Component 1:** Fixed DWT layer (db1) → 3 branches
# 2. **Component 2:** CNN autoencoder per branch with branch-specific pooling
# 3. **Component 3:** LSTM classifier over concatenated branch features

# %% [markdown]
## 1. Setup & Data Loading
# %%
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import scipy.io as sio
import pywt

# reproducibility and device
torch.manual_seed(42)
device = torch.device(
                    "cuda"
                    if torch.cuda.is_available()
                    else "mps" if torch.backends.mps.is_available() else "cpu"
                )
print(f"Using {device}")

# load & preprocess
data = sio.loadmat('../data/mine_impact_data_2019.mat')
samps = pd.DataFrame(data['x'].T)
labs  = pd.DataFrame(data['y'].T, columns=['y'])
df = pd.concat([samps, labs], axis=1).dropna().sample(frac=1, random_state=42)
X = df.iloc[:, :-1].values.astype(np.float32)
y = df.iloc[:,  -1].values.astype(np.int64)
X = (X - X.min()) / (X.max() - X.min())

# tensors and split
X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)
train_n = 3000
train_X, test_X = X_t[:train_n], X_t[train_n:]
train_y, test_y = y_t[:train_n], y_t[train_n:]

# dataloaders
bs=30
train_ds = TensorDataset(train_X.unsqueeze(1), train_y)
test_ds  = TensorDataset(test_X.unsqueeze(1),  test_y)
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False)

# %% [markdown]
## 2. Component 1: DWT Layer
# %%
class DWT_1D(nn.Module):
    def __init__(self, wavename='db1'):
        super().__init__()
        w = pywt.Wavelet(wavename)
        lo = torch.tensor(w.dec_lo[::-1], dtype=torch.float32)
        hi = torch.tensor(w.dec_hi[::-1], dtype=torch.float32)
        self.register_buffer('lo', lo.view(1,1,-1))
        self.register_buffer('hi', hi.view(1,1,-1))
    def forward(self, x):
        L = F.conv1d(x, self.lo, stride=2)
        H = F.conv1d(x, self.hi, stride=2)
        return L, H

class WaveletFirstBlock(nn.Module):
    def __init__(self, wavename='db1'):
        super().__init__()
        self.dwt1 = DWT_1D(wavename)
        self.dwt2 = DWT_1D(wavename)
    def forward(self, x):
        L1, H1 = self.dwt1(x)
        L2, H2 = self.dwt2(L1)
        return L2, H1, H2

# %% [markdown]
## 3. Component 2: Branch‐specific CNN Autoencoder
# %%
class ConvBranch(nn.Module):
    def __init__(self, pool_sizes):
        super().__init__()
        chans = [1,2,5,10]
        layers=[]
        for i,p in enumerate(pool_sizes):
            layers += [nn.Conv1d(chans[i], chans[i+1], kernel_size=3, padding=1),
                       nn.Tanh(), nn.MaxPool1d(p)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DeConvBranch(nn.Module):
    def __init__(self, pool_sizes):
        super().__init__()
        chans = [10,5,2,1]
        layers=[]
        for i, up in enumerate(pool_sizes):
            layers += [nn.Upsample(scale_factor=up, mode='nearest'),
                       nn.ConvTranspose1d(chans[i], chans[i+1], kernel_size=3, padding=1)]
            # last layer linear, earlier add Tanh
            if i < len(pool_sizes)-1:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DWTNetAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.dwt = WaveletFirstBlock('db1')
        # branch pool schedules produce equal T_enc
        self.enc = nn.ModuleList([
            ConvBranch([4,4,4]),  # Approx-2: 9000->2250->562->140
            ConvBranch([8,4,4]),  # Detail-1:18000->2250->562->140
            ConvBranch([4,4,4]),  # Detail-2:9000->2250->562->140
        ])
        self.dec = nn.ModuleList([
            DeConvBranch([4,4,4]),
            DeConvBranch([4,4,4]),
            DeConvBranch([4,4,4]),
        ])
    def forward(self, x):
        L2, H1, H2 = self.dwt(x)
        e2 = self.enc[0](L2)
        d1 = self.enc[1](H1)
        d2 = self.enc[2](H2)
        # decode
        r2  = self.dec[0](e2)
        rd1 = self.dec[1](d1)
        rd2 = self.dec[2](d2)
        return (e2, d1, d2), (r2, rd1, rd2)

# train autoencoder
ae = DWTNetAE().to(device)
opt_ae = optim.Adam(ae.parameters(), lr=1e-3)
crit_ae = nn.MSELoss()
epochs_ae = 10
for ep in range(1, epochs_ae+1):
    ae.train(); tot=0
    for x,_ in train_loader:
        x = x.to(device)
        (e2,d1,d2), (r2,rd1,rd2) = ae(x)
        with torch.no_grad(): t2, td1, td2 = ae.dwt(x)
        # align r* to t*
        r2  = F.interpolate(r2, size=t2.shape[-1], mode='nearest')
        rd1 = F.interpolate(rd1, size=td1.shape[-1], mode='nearest')
        rd2 = F.interpolate(rd2, size=td2.shape[-1], mode='nearest')
        loss = crit_ae(r2,t2) + crit_ae(rd1,td1) + crit_ae(rd2,td2)
        opt_ae.zero_grad(); loss.backward(); opt_ae.step()
        tot += loss.item()*x.size(0)
    print(f"[AE] Ep{ep:02d} Loss={tot/train_n:.6f}")

# %% [markdown]
## 4. Component 3: LSTM Classifier
# %%
class ClassifierNet(nn.Module):
    def __init__(self, ae):
        super().__init__()
        self.dwt = ae.dwt
        self.enc = ae.enc
        for p in self.dwt.parameters(): p.requires_grad=False
        for m in self.enc:  
            for p in m.parameters(): p.requires_grad=False
        # input_size = sum of channels from all branches = 10+10+10 = 30
        self.lstm = nn.LSTM(input_size=30, hidden_size=100, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(100,50), nn.Tanh(), nn.Dropout(0.5), nn.Linear(50,2))
    def forward(self, x):
        L2,H1,H2 = self.dwt(x)
        e2 = self.enc[0](L2)
        d1 = self.enc[1](H1)
        d2 = self.enc[2](H2)
        z = torch.cat([e2,d1,d2], dim=1)  # (B,30,T)
        seq = z.transpose(1,2)            # (B,T,30)
        out,_ = self.lstm(seq)
        h = out[:,-1,:]
        return self.fc(h)

clf = ClassifierNet(ae).to(device)
opt_clf = optim.Adam(filter(lambda p: p.requires_grad, clf.parameters()), lr=1e-3)
crit_clf = nn.CrossEntropyLoss()
epochs_clf = 10
for ep in range(1, epochs_clf+1):
    clf.train(); tot=0; corr=0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        logits = clf(x)
        loss = crit_clf(logits,y)
        opt_clf.zero_grad(); loss.backward(); opt_clf.step()
        tot += loss.item()*x.size(0)
        corr+= (logits.argmax(1)==y).sum().item()
    print(f"[CLF] Ep{ep:02d} Loss={tot/train_n:.4f} Acc={corr/train_n:.4f}")

# %% [markdown]
## 5. Final Test Evaluation
# %%
clf.eval(); allp=[]; ally=[]
with torch.no_grad():
    for x,y in test_loader:
        p = clf(x.to(device)).argmax(1).cpu()
        allp.append(p); ally.append(y)
    allp = torch.cat(allp); ally = torch.cat(ally)
    print(f"Final Test Accuracy: { (allp==ally).float().mean().item() }")


Using mps
[AE] Ep01 Loss=1.355923
[AE] Ep02 Loss=0.009133
[AE] Ep03 Loss=0.000998
[AE] Ep04 Loss=0.000992
[AE] Ep05 Loss=0.000986
[AE] Ep06 Loss=0.000979
[AE] Ep07 Loss=0.000972
[AE] Ep08 Loss=0.000964
[AE] Ep09 Loss=0.000957
[AE] Ep10 Loss=0.000949
[CLF] Ep01 Loss=0.6458 Acc=0.6620
[CLF] Ep02 Loss=0.6420 Acc=0.6657
[CLF] Ep03 Loss=0.6410 Acc=0.6657
[CLF] Ep04 Loss=0.6392 Acc=0.6657
[CLF] Ep05 Loss=0.6411 Acc=0.6657
[CLF] Ep06 Loss=0.6395 Acc=0.6657
[CLF] Ep07 Loss=0.6405 Acc=0.6657
[CLF] Ep08 Loss=0.6398 Acc=0.6657
[CLF] Ep09 Loss=0.6413 Acc=0.6657
[CLF] Ep10 Loss=0.6392 Acc=0.6657
Final Test Accuracy: 0.6957928538322449


In [3]:
# %% [markdown]
# Feature Extraction and Classification of Acoustic Signals (Simplified)

# This notebook implements a three-component network adapted for binary mine-impact classification with a reduced-depth CNN autoencoder (per-branch channels 1→2→5).

# %% [markdown]
## 1. Setup & Data Loading
# %%
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import scipy.io as sio
import pywt

# reproducibility and device
torch.manual_seed(42)
device = torch.device(
                    "cuda"
                    if torch.cuda.is_available()
                    else "mps" if torch.backends.mps.is_available() else "cpu"
                )

print(f"Using {device}")

# Load and preprocess data
data = sio.loadmat('../data/mine_impact_data_2019.mat')
samps = pd.DataFrame(data['x'].T)
labs = pd.DataFrame(data['y'].T, columns=['y'])
df = pd.concat([samps, labs], axis=1).dropna().sample(frac=1, random_state=42)
X = df.iloc[:, :-1].values.astype(np.float32)
y = df.iloc[:, -1].values.astype(np.int64)
# Normalize
# X = (X - X.min()) / (X.max() - X.min())

# Convert to tensors and split
total = X.shape[0]
train_n = 3000
test_n = total - train_n
X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)
train_X, test_X = X_t[:train_n], X_t[train_n:]
train_y, test_y = y_t[:train_n], y_t[train_n:]

# DataLoaders (add channel dim)
bs = 30
train_ds = TensorDataset(train_X.unsqueeze(1), train_y)
test_ds = TensorDataset(test_X.unsqueeze(1), test_y)
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False)

# %% [markdown]
## 2. Component 1: DWT Layer
# %%
class DWT_1D(nn.Module):
    def __init__(self, wavename='db1'):
        super().__init__()
        w = pywt.Wavelet(wavename)
        lo = torch.tensor(w.dec_lo[::-1], dtype=torch.float32)
        hi = torch.tensor(w.dec_hi[::-1], dtype=torch.float32)
        self.register_buffer('lo', lo.view(1,1,-1))
        self.register_buffer('hi', hi.view(1,1,-1))
    def forward(self, x):
        L = F.conv1d(x, self.lo, stride=2)
        H = F.conv1d(x, self.hi, stride=2)
        return L, H

class WaveletFirstBlock(nn.Module):
    def __init__(self, wavename='db1'):
        super().__init__()
        self.dwt1 = DWT_1D(wavename)
        self.dwt2 = DWT_1D(wavename)
    def forward(self, x):
        L1, H1 = self.dwt1(x)
        L2, H2 = self.dwt2(L1)
        return L2, H1, H2

# %% [markdown]
## 3. Component 2: Simplified CNN Autoencoder (1→2→5 Channels)
# %%
class ConvBranch(nn.Module):
    def __init__(self, pool_sizes):
        super().__init__()
        chans = [1, 2, 5]
        layers = []
        for i, p in enumerate(pool_sizes):
            layers += [
                nn.Conv1d(chans[i], chans[i+1], kernel_size=3, padding=1),
                nn.Tanh(),
                nn.MaxPool1d(p)
            ]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class DeConvBranch(nn.Module):
    def __init__(self, pool_sizes):
        super().__init__()
        chans = [5, 2, 1]
        layers = []
        for i, up in enumerate(pool_sizes):
            layers += [
                nn.Upsample(scale_factor=up, mode='nearest'),
                nn.ConvTranspose1d(chans[i], chans[i+1], kernel_size=3, padding=1)
            ]
            if i < len(pool_sizes) - 1:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class DWTNetAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.dwt = WaveletFirstBlock('db1')
        # branch pool schedules: [4,4] for Approx & Detail-2, [8,4] for Detail-1
        self.enc = nn.ModuleList([
            ConvBranch([4,4]),  # Approx-2
            ConvBranch([8,4]),  # Detail-1
            ConvBranch([4,4])   # Detail-2
        ])
        self.dec = nn.ModuleList([
            DeConvBranch([4,4]),    # reverse enc on Approx-2
            DeConvBranch([4,8]),    # reverse enc on Detail-1
            DeConvBranch([4,4])     # reverse enc on Detail-2
        ])
    def forward(self, x):
        L2, H1, H2 = self.dwt(x)
        e2 = self.enc[0](L2)
        d1 = self.enc[1](H1)
        d2 = self.enc[2](H2)
        r2 = self.dec[0](e2)
        rd1 = self.dec[1](d1)
        rd2 = self.dec[2](d2)
        return (e2, d1, d2), (r2, rd1, rd2)

# Train the autoencoder
ae = DWTNetAE().to(device)
opt_ae = optim.Adam(ae.parameters(), lr=1e-4, weight_decay=1e-5)
crit_ae = nn.MSELoss()
epochs_ae = 10
for ep in range(1, epochs_ae+1):
    ae.train(); total_loss = 0
    for x, _ in train_loader:
        x = x.to(device)
        (e2, d1, d2), (r2, rd1, rd2) = ae(x)
        with torch.no_grad():
            t2, td1, td2 = ae.dwt(x)
        # align shapes via interpolation
        r2 = F.interpolate(r2, size=t2.shape[-1], mode='nearest')
        rd1 = F.interpolate(rd1, size=td1.shape[-1], mode='nearest')
        rd2 = F.interpolate(rd2, size=td2.shape[-1], mode='nearest')
        loss = crit_ae(r2, t2) + crit_ae(rd1, td1) + crit_ae(rd2, td2)
        opt_ae.zero_grad(); loss.backward(); opt_ae.step()
        total_loss += loss.item() * x.size(0)
    print(f"[AE] Epoch {ep:02d} Loss: {total_loss/train_n:.6f}")

# %% [markdown]
## 4. Component 3: LSTM Classifier
# %%
class ClassifierNet(nn.Module):
    def __init__(self, ae):
        super().__init__()
        self.dwt = ae.dwt
        self.enc = ae.enc
        # freeze DWT and encoder
        for p in self.dwt.parameters(): p.requires_grad=False
        for m in self.enc:
            for p in m.parameters(): p.requires_grad=False
        # total encoded channels = 5+5+5 = 15
        self.lstm = nn.LSTM(input_size=15, hidden_size=100, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(100, 50), nn.Tanh(), nn.Dropout(0.5), nn.Linear(50, 2)
        )
    def forward(self, x):
        L2, H1, H2 = self.dwt(x)
        e2 = self.enc[0](L2)
        d1 = self.enc[1](H1)
        d2 = self.enc[2](H2)
        z = torch.cat([e2, d1, d2], dim=1)  # (B,15,T)

        seq = z.transpose(1, 2)           # (B,T,15)
        out, _ = self.lstm(seq)
        h = out[:, -1, :]
        return self.fc(h)

clf = ClassifierNet(ae).to(device)
opt_clf = optim.Adam(filter(lambda p: p.requires_grad, clf.parameters()), lr=1e-3, weight_decay=1e-5)
crit_clf = nn.CrossEntropyLoss()
epochs_clf = 5
for ep in range(1, epochs_clf+1):
    clf.train(); total_loss = 0; correct = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = clf(x)
        loss = crit_clf(logits, y)
        opt_clf.zero_grad(); loss.backward(); opt_clf.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
    print(f"[CLF] Epoch {ep:02d} Loss={total_loss/train_n:.4f} Acc={correct/train_n:.4f}")

# %% [markdown]
## 5. Final Test Evaluation
# %%
clf.eval(); preds = []; trues = []
with torch.no_grad():
    for x, y in test_loader:
        p = clf(x.to(device)).argmax(1).cpu()
        preds.append(p); trues.append(y)
    preds = torch.cat(preds); trues = torch.cat(trues)
    acc = (preds == trues).float().mean().item()
print(f"Final Test Accuracy: {acc:.4f}")


Using mps
[AE] Epoch 01 Loss: 0.390158
[AE] Epoch 02 Loss: 0.264449
[AE] Epoch 03 Loss: 0.172960
[AE] Epoch 04 Loss: 0.103964
[AE] Epoch 05 Loss: 0.056016
[AE] Epoch 06 Loss: 0.026731
[AE] Epoch 07 Loss: 0.011241
[AE] Epoch 08 Loss: 0.004370
[AE] Epoch 09 Loss: 0.001888
[AE] Epoch 10 Loss: 0.001161
[CLF] Epoch 01 Loss=0.6464 Acc=0.6550
[CLF] Epoch 02 Loss=0.6441 Acc=0.6657
[CLF] Epoch 03 Loss=0.6400 Acc=0.6657
[CLF] Epoch 04 Loss=0.6406 Acc=0.6657
[CLF] Epoch 05 Loss=0.6412 Acc=0.6657
Final Test Accuracy: 0.6958


In [62]:
from sklearn.metrics import confusion_matrix, classification_report

clf.eval()
with torch.no_grad():
    # Add channel dim: (N, L) → (N, 1, L)
    inputs = test_X.unsqueeze(1).to(device)
    logits = clf(inputs)
    preds = logits.argmax(dim=1).cpu().numpy()
    trues = test_y.numpy()

# Confusion matrix
cm = confusion_matrix(trues, preds)
print("Confusion Matrix:\n", cm)

# Detailed report
print("\nClassification Report:\n", 
      classification_report(trues, preds, target_names=['drummy (0)', 'tight (1)']))

Confusion Matrix:
 [[215   0]
 [ 94   0]]

Classification Report:
               precision    recall  f1-score   support

  drummy (0)       0.70      1.00      0.82       215
   tight (1)       0.00      0.00      0.00        94

    accuracy                           0.70       309
   macro avg       0.35      0.50      0.41       309
weighted avg       0.48      0.70      0.57       309



/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kanwarsingh/Desktop/AI/acoustic_signal_classification/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_div